<a href="https://colab.research.google.com/github/rauf-mifteev/Ahuntsic_AI_Optimization/blob/main/AI_TP_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intelligence Artificielle 1 — Travail Pratique 2

Optimisation : Programmation Linéaire, Glouton et Programmation Dynamique

## Partie 1 — Programmation linéaire avec PuLP

1.1 Données du problème

In [1]:
joueurs = [
    {"nom": "Alice", "score": 88, "salaire": 1200, "poids": 72},
    {"nom": "Bob", "score": 91, "salaire": 1800, "poids": 85},
    {"nom": "Clara", "score": 84, "salaire": 950, "poids": 68},
    {"nom": "David", "score": 93, "salaire": 2100, "poids": 90},
    {"nom": "Emma", "score": 79, "salaire": 300, "poids": 65},
    {"nom": "Frank", "score": 87, "salaire": 2400, "poids": 95},
    {"nom": "Grace", "score": 85, "salaire": 1050, "poids": 70},
    {"nom": "Hugo", "score": 89, "salaire": 1600, "poids": 80}
]

1.2 Définition du problème de maximisation

In [2]:
import pulp

prob = pulp.LpProblem("Ligue_Basketball", pulp.LpMaximize)

1.3 Variables de décision binaires (0 ou 1)

In [3]:
# x_A[nom] = 1 si le joueur est dans l'équipe A
x_A = pulp.LpVariable.dicts("EqA", [j["nom"] for j in joueurs], cat='Binary')
x_B = pulp.LpVariable.dicts("EqB", [j["nom"] for j in joueurs], cat='Binary')

1.4 Fonction objective : Maximiser le score total

In [4]:
prob += pulp.lpSum([j["score"] * (x_A[j["nom"]] + x_B[j["nom"]]) for j in joueurs]), "Score_Total"

1.5 Contraintes

In [5]:
# Exactement 3 joueurs par équipe
prob += pulp.lpSum([x_A[j["nom"]] for j in joueurs]) == 3, "Taille_EqA"
prob += pulp.lpSum([x_B[j["nom"]] for j in joueurs]) == 3, "Taille_EqB"

# Un joueur ne peut être que dans une seule équipe (ou aucune)
for j in joueurs:
    prob += x_A[j["nom"]] + x_B[j["nom"]] <= 1, f"Unicite_{j['nom']}"

# Budget total max de 8500$
prob += pulp.lpSum([j["salaire"] * (x_A[j["nom"]] + x_B[j["nom"]]) for j in joueurs]) <= 8500, "Budget_Total"

# Poids max de 250 Kg par équipe
prob += pulp.lpSum([j["poids"] * x_A[j["nom"]] for j in joueurs]) <= 250, "Poids_EqA"
prob += pulp.lpSum([j["poids"] * x_B[j["nom"]] for j in joueurs]) <= 250, "Poids_EqB"

1.6 Résolution

In [6]:
prob.solve()

1

1.7 Affichage des résultats

In [7]:
print(f"Statut : {pulp.LpStatus[prob.status]}\n")

equipe_A, equipe_B = [], []
score_A, score_B = 0, 0

for j in joueurs:
    if x_A[j["nom"]].varValue == 1:
        equipe_A.append(j["nom"])
        score_A += j["score"]
    elif x_B[j["nom"]].varValue == 1:
        equipe_B.append(j["nom"])
        score_B += j["score"]

print(f"Équipe A : {equipe_A} (Score: {score_A})")
print(f"Équipe B : {equipe_B} (Score: {score_B})")
print(f"Score Total Optimal : {pulp.value(prob.objective)} points")

Statut : Optimal

Équipe A : ['Emma', 'Grace', 'Hugo'] (Score: 253)
Équipe B : ['Alice', 'Bob', 'David'] (Score: 272)
Score Total Optimal : 525.0 points
